In [77]:
from pathlib import Path
import gcamreader
import os
import pandas as pd
import numpy as np
from utils import convert_to_mt, ej_to_twh
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

In [78]:
ej_to_twh(0.554)

153.88888901200002

In [79]:
for t in range(5, 21, 5):
    f = 1 / (1 + np.exp(0.1 * (t-22.5)))
    print(t, f)

5 0.8519528019683106
10 0.7772998611746911
15 0.679178699175393
20 0.5621765008857981


In [80]:
def to_Mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100-yr GWP
GWP_AR5 = {
    'CO2':    1,
    'CH4':    28,
    'N2O':    265,
    'HFC125': 3500,
    'HFC134a':1430,
    'HFC143a':4470,
    'HFC23':  14800,
    'HFC32':  675,
    'HFC43':  1500,
    'HFC227ea':3220,
    'HFC236fa':9810,
    'SF6':    23500,
    'C2F6':   12200,
    'CF4':    6630,
}

def ej_to_twh(ej):
    """
    Convert energy from exajoules (EJ) to terawatt-hours (TWh).

    Parameters:
    ej (float): Energy in exajoules.

    Returns:
    float: Energy in terawatt-hours.
    """
    twh = ej * 277.777778
    return twh

# Define custom colors for each class
custom_colors = {
    'Solar': '#FECB52',  # Yellow
    'Wind': 'rgb(136,204,238)',  # Light blue
    'Hydro': 'rgb(95, 70, 144)',  # Dark blue
    'Nuclear': '#AB63FA',  # Orange
    'Biomass': 'rgb(115, 175, 72)',  # Dark green
    'Gas w/ CCS': '#DEA0FD',     # Pink
    'Gas': '#FFA15A',    # Purple
    'Coal w/ CCS': '#750D86',    # Maroon
    'Coal': '#222A2A',   # Red
    'Others': 'rgb(217,217,217)',      # Dark orange
    'Hydrogen': "#727DCD",
    'Ammonia': "rgb(231,63,116)",
    'Oil': "#7D1215",
    'Oil w/ CCS': "#8D5757",
}

In [81]:
proj_path = Path("/data/project/tae/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [82]:
dbpath = "../output/"  # relative to current working directory
dbfile = "database_basexdb_korea_2035_20250721_7"
conn = gcamreader.LocalDBConn(dbpath, dbfile)
queries = gcamreader.parse_batch_query(os.path.join('..', 'output', 'queries','Main_queries.xml'))

Database scenarios: Current-Policy, Current-Policy, Enhanced-Ambition


In [83]:
scenarios = list(conn.listScenariosInDB()['name'])
scenarios

['Current-Policy', 'Current-Policy', 'Enhanced-Ambition']

In [84]:
scenarios = ['Current-Policy', 'Enhanced-Ambition']

In [85]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [86]:
i = 9
q = queries[i]
print(q.title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
# df['vint'] = df['technology'].str.split('=').str[1].astype(int)
df.head()

elec gen by gen tech


,Units,scenario,region,subsector,technology,output,Year,value
0,EJ,Current-Policy,South Korea,biomass,biomass (IGCC),electricity,2025,0.000735
1,EJ,Current-Policy,South Korea,biomass,biomass (IGCC),electricity,2030,0.002086
2,EJ,Current-Policy,South Korea,biomass,biomass (IGCC),electricity,2035,0.004201
3,EJ,Current-Policy,South Korea,biomass,biomass (conv),electricity,2005,0.000162
4,EJ,Current-Policy,South Korea,biomass,biomass (conv),electricity,2010,0.001246


In [87]:
df['technology'].unique()

array(['biomass (IGCC)', 'biomass (conv)', 'coal (IGCC CCS)',
       'coal (conv pul CCS)', 'coal (conv pul ammonia blend 20%)',
       'coal (conv pul)', 'gas (CC CCS)', 'gas (CC H2 blend 50%)',
       'gas (CC)', 'gas (steam/CT)', 'hydro', 'Gen_III', 'Gen_II_LWR',
       'refined liquids (CC)', 'refined liquids (steam/CT)', 'rooftop_pv',
       'PV', 'PV_storage', 'wind', 'wind_offshore', 'wind_storage'],
      dtype=object)

In [88]:
df[(df['technology'] == 'rooftop_pv') & (df['Year'] >= 2015)]

,Units,scenario,region,subsector,technology,output,Year,value
80,EJ,Current-Policy,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2020,0.035924
81,EJ,Current-Policy,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2025,0.047784
82,EJ,Current-Policy,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2030,0.047267
83,EJ,Current-Policy,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2035,0.058048
189,EJ,Enhanced-Ambition,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2020,0.035908
190,EJ,Enhanced-Ambition,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2025,0.047954
191,EJ,Enhanced-Ambition,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2030,0.059410
192,EJ,Enhanced-Ambition,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2035,0.112471


In [40]:
i = 314
q = queries[i]
print(q.title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
# df['vint'] = df['technology'].str.split('=').str[1].astype(int)
df.head()

prices of all markets


,Units,scenario,Year,market,value
0,$/GJ,Current-Policy,1975,South Koreawoodpulp_energy,6.92383
1,$/GJ,Current-Policy,1990,South Koreawoodpulp_energy,98.45780
2,$/GJ,Current-Policy,2005,South Koreawoodpulp_energy,51.64210
3,$/GJ,Current-Policy,2010,South Koreawoodpulp_energy,45.54300
4,$/GJ,Current-Policy,2015,South Koreawoodpulp_energy,26.85570


In [41]:
df[(df['market'] == 'South KoreaOnwind-Generation-Ceiling')]

,Units,scenario,Year,market,value
147,1975$/GJ,Current-Policy,1975,South KoreaOnwind-Generation-Ceiling,0.00000
315,1975$/GJ,Current-Policy,1990,South KoreaOnwind-Generation-Ceiling,0.00000
483,1975$/GJ,Current-Policy,2005,South KoreaOnwind-Generation-Ceiling,0.00000
651,1975$/GJ,Current-Policy,2010,South KoreaOnwind-Generation-Ceiling,0.00000
819,1975$/GJ,Current-Policy,2015,South KoreaOnwind-Generation-Ceiling,0.00000
987,1975$/GJ,Current-Policy,2020,South KoreaOnwind-Generation-Ceiling,8.82717
1155,1975$/GJ,Current-Policy,2025,South KoreaOnwind-Generation-Ceiling,14.56330
1323,1975$/GJ,Current-Policy,2030,South KoreaOnwind-Generation-Ceiling,6.35764
1491,1975$/GJ,Current-Policy,2035,South KoreaOnwind-Generation-Ceiling,5.46434
1659,1975$/GJ,Current-Policy,2040,South KoreaOnwind-Generation-Ceiling,0.00000


In [42]:
df['market'].unique()

array(['South Koreawoodpulp_energy', 'South Koreasawnwood_processing',
       'South Koreawoodpulp_processing',
       'South KoreaCoal-Ammonia-Blend-Ceiling',
       'South KoreaCoal-Generation-Ceiling',
       'South KoreaGas-Generation-Ceiling',
       'South KoreaGas-H2-Blend-Floor',
       'South KoreaH2 central production', 'South KoreaH2 industrial',
       'South KoreaH2 liquid truck', 'South KoreaH2 pipeline',
       'South KoreaH2 retail delivery', 'South KoreaH2 retail dispensing',
       'South KoreaH2 wholesale delivery',
       'South KoreaH2 wholesale dispensing', 'South KoreaImport-Ceiling',
       'South KoreaNuclear-Ceiling',
       'South KoreaOffwind-Generation-Floor',
       'South KoreaOnwind-Generation-Ceiling', 'South KoreaScrap-Ceiling',
       'South KoreaSolar-Generation-Floor',
       'South Koreaagricultural energy use',
       'South Koreabackup_electricity', 'South Koreabiomass',
       'South Koreachemical', 'South Koreachemical energy use',
       'Sout

In [43]:
df = conn.runQuery(queries[21], scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
# df['vint'] = df['technology'].str.split('=').str[1].astype(int)
df.head()

,Units,scenario,region,sector,subsector,Year,technology,value
0,None Specified,Current-Policy,South Korea,elect_td_bld,rooftop_pv,2020,rooftop_pv,1.0
1,None Specified,Current-Policy,South Korea,elect_td_bld,rooftop_pv,2025,rooftop_pv,1.0
2,None Specified,Current-Policy,South Korea,elect_td_bld,rooftop_pv,2030,rooftop_pv,1.0
3,None Specified,Current-Policy,South Korea,elect_td_bld,rooftop_pv,2035,rooftop_pv,1.0
4,None Specified,Current-Policy,South Korea,elect_td_bld,rooftop_pv,2040,rooftop_pv,1.0


In [44]:
df[(df['Year'] == 2025)]

,Units,scenario,region,sector,subsector,Year,technology,value
1,None Specified,Current-Policy,South Korea,elect_td_bld,rooftop_pv,2025,rooftop_pv,1.000000
21,None Specified,Current-Policy,South Korea,electricity,biomass,2025,biomass (IGCC),1.000000
22,None Specified,Current-Policy,South Korea,electricity,biomass,2025,biomass (conv),1.000000
78,None Specified,Current-Policy,South Korea,electricity,coal,2025,coal (IGCC CCS),1.000000
79,None Specified,Current-Policy,South Korea,electricity,coal,2025,coal (conv pul CCS),1.000000
155,None Specified,Current-Policy,South Korea,electricity,gas,2025,gas (CC CCS),1.000000
156,None Specified,Current-Policy,South Korea,electricity,gas,2025,gas (CC H2 blend 50%),1.000000
157,None Specified,Current-Policy,South Korea,electricity,gas,2025,gas (CC),1.000000
158,None Specified,Current-Policy,South Korea,electricity,gas,2025,gas (steam/CT),0.251066
220,None Specified,Current-Policy,South Korea,electricity,geothermal,2025,geothermal,1.000000


In [45]:
df = conn.runQuery(queries[11], scenarios=scenarios[-2:], regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df['vint'] = df['technology'].str.split('=').str[1].astype(int)
df.head()

,Units,scenario,region,sector,subsector,technology,output,Year,value,vint
0,EJ,Current-Policy,South Korea,elect_td_bld,rooftop_pv,"rooftop_pv,year=2020",elect_td_bld,2020,0.035924,2020
1,EJ,Current-Policy,South Korea,elect_td_bld,rooftop_pv,"rooftop_pv,year=2025",elect_td_bld,2025,0.047784,2025
2,EJ,Current-Policy,South Korea,elect_td_bld,rooftop_pv,"rooftop_pv,year=2030",elect_td_bld,2030,0.047267,2030
3,EJ,Current-Policy,South Korea,elect_td_bld,rooftop_pv,"rooftop_pv,year=2035",elect_td_bld,2035,0.058048,2035
4,EJ,Current-Policy,South Korea,electricity,biomass,"biomass (IGCC) (dry cooling),year=2025",elec_biomass (IGCC),2025,0.000016,2025


In [46]:
df[(df['scenario'] == 'Current-Policy') & (df['Year'] == 2035) & (df['subsector'] == 'refined liquids')]#['value'].sum()

,Units,scenario,region,sector,subsector,technology,output,Year,value,vint
334,EJ,Current-Policy,South Korea,electricity,refined liquids,"refined liquids (CC) (dry cooling),year=2020",elec_refined liquids (CC),2035,0.000011,2020
337,EJ,Current-Policy,South Korea,electricity,refined liquids,"refined liquids (CC) (dry cooling),year=2025",elec_refined liquids (CC),2035,0.000022,2025
339,EJ,Current-Policy,South Korea,electricity,refined liquids,"refined liquids (CC) (dry cooling),year=2030",elec_refined liquids (CC),2035,0.000047,2030
340,EJ,Current-Policy,South Korea,electricity,refined liquids,"refined liquids (CC) (dry cooling),year=2035",elec_refined liquids (CC),2035,0.000423,2035
344,EJ,Current-Policy,South Korea,electricity,refined liquids,"refined liquids (CC) (recirculating),year=2020",elec_refined liquids (CC),2035,0.000455,2020
347,EJ,Current-Policy,South Korea,electricity,refined liquids,"refined liquids (CC) (recirculating),year=2025",elec_refined liquids (CC),2035,0.000873,2025
349,EJ,Current-Policy,South Korea,electricity,refined liquids,"refined liquids (CC) (recirculating),year=2030",elec_refined liquids (CC),2035,0.001847,2030
350,EJ,Current-Policy,South Korea,electricity,refined liquids,"refined liquids (CC) (recirculating),year=2035",elec_refined liquids (CC),2035,0.013430,2035
354,EJ,Current-Policy,South Korea,electricity,refined liquids,"refined liquids (CC) (seawater),year=2020",elec_refined liquids (CC),2035,0.000122,2020
357,EJ,Current-Policy,South Korea,electricity,refined liquids,"refined liquids (CC) (seawater),year=2025",elec_refined liquids (CC),2035,0.000234,2025


In [47]:
df[(df['Year'] == 2035)]

,Units,scenario,region,sector,subsector,technology,output,Year,value,vint
3,EJ,Current-Policy,South Korea,elect_td_bld,rooftop_pv,"rooftop_pv,year=2035",elect_td_bld,2035,0.058048,2035
6,EJ,Current-Policy,South Korea,electricity,biomass,"biomass (IGCC) (dry cooling),year=2025",elec_biomass (IGCC),2035,0.000012,2025
8,EJ,Current-Policy,South Korea,electricity,biomass,"biomass (IGCC) (dry cooling),year=2030",elec_biomass (IGCC),2035,0.000025,2030
9,EJ,Current-Policy,South Korea,electricity,biomass,"biomass (IGCC) (dry cooling),year=2035",elec_biomass (IGCC),2035,0.000054,2035
12,EJ,Current-Policy,South Korea,electricity,biomass,"biomass (IGCC) (recirculating),year=2025",elec_biomass (IGCC),2035,0.000445,2025
...,...,...,...,...,...,...,...,...,...,...
962,EJ,Enhanced-Ambition,South Korea,electricity,wind,"wind_offshore,year=2035",electricity,2035,0.250099,2035
966,EJ,Enhanced-Ambition,South Korea,electricity,wind,"wind_storage,year=2020",electricity,2035,0.000230,2020
969,EJ,Enhanced-Ambition,South Korea,electricity,wind,"wind_storage,year=2025",electricity,2035,0.000213,2025
971,EJ,Enhanced-Ambition,South Korea,electricity,wind,"wind_storage,year=2030",electricity,2035,0.000670,2030


In [48]:
0.309-0.133

0.176

In [49]:
0.176 / 0.43

0.40930232558139534

In [50]:
df[(df['subsector'] == 'gas') & (df['output'].isin(['elec_gas (CC)', 'elec_gas (steam/CT)']))].groupby(['Year', 'scenario', 'vint'])['value'].sum()

Year  scenario           vint
1990  Current-Policy     1990    0.034576
      Enhanced-Ambition  1990    0.034576
2005  Current-Policy     2005    0.213065
      Enhanced-Ambition  2005    0.213065
2010  Current-Policy     2010    0.365474
      Enhanced-Ambition  2010    0.365474
2015  Current-Policy     2015    0.433323
      Enhanced-Ambition  2015    0.433323
2020  Current-Policy     2015    0.424924
                         2020    0.101076
      Enhanced-Ambition  2015    0.424915
                         2020    0.101085
2025  Current-Policy     2015    0.429316
                         2020    0.100265
                         2025    0.033419
      Enhanced-Ambition  2015    0.429453
                         2020    0.100301
                         2025    0.033246
2030  Current-Policy     2015    0.401624
                         2020    0.098931
                         2025    0.032750
                         2030    0.020710
      Enhanced-Ambition  2015    0.404265
    

In [51]:
0.156 / 0.433323

0.36000858482009956

In [52]:
0.01 / 0.433323

0.023077473385903817

In [89]:
df = conn.runQuery(queries[9], scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df.head()

,Units,scenario,region,subsector,technology,output,Year,value
0,EJ,Current-Policy,South Korea,biomass,biomass (IGCC),electricity,2025,0.000735
1,EJ,Current-Policy,South Korea,biomass,biomass (IGCC),electricity,2030,0.002086
2,EJ,Current-Policy,South Korea,biomass,biomass (IGCC),electricity,2035,0.004201
3,EJ,Current-Policy,South Korea,biomass,biomass (conv),electricity,2005,0.000162
4,EJ,Current-Policy,South Korea,biomass,biomass (conv),electricity,2010,0.001246


In [90]:
df['technology'].unique()

array(['biomass (IGCC)', 'biomass (conv)', 'coal (IGCC CCS)',
       'coal (conv pul CCS)', 'coal (conv pul ammonia blend 20%)',
       'coal (conv pul)', 'gas (CC CCS)', 'gas (CC H2 blend 50%)',
       'gas (CC)', 'gas (steam/CT)', 'hydro', 'Gen_III', 'Gen_II_LWR',
       'refined liquids (CC)', 'refined liquids (steam/CT)', 'rooftop_pv',
       'PV', 'PV_storage', 'wind', 'wind_offshore', 'wind_storage'],
      dtype=object)

In [91]:
df[(df['technology'] == 'rooftop_pv')]

,Units,scenario,region,subsector,technology,output,Year,value
80,EJ,Current-Policy,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2020,0.035924
81,EJ,Current-Policy,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2025,0.047784
82,EJ,Current-Policy,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2030,0.047267
83,EJ,Current-Policy,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2035,0.058048
189,EJ,Enhanced-Ambition,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2020,0.035908
190,EJ,Enhanced-Ambition,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2025,0.047954
191,EJ,Enhanced-Ambition,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2030,0.059410
192,EJ,Enhanced-Ambition,South Korea,rooftop_pv,rooftop_pv,elect_td_bld,2035,0.112471


In [92]:
df[(df['Year'] == 2035) & (df['subsector'] == 'solar')]#.groupby(['scenario'])['value'].sum()

,Units,scenario,region,subsector,technology,output,Year,value
91,EJ,Current-Policy,South Korea,solar,PV,electricity,2035,0.386311
95,EJ,Current-Policy,South Korea,solar,PV_storage,electricity,2035,0.001093
200,EJ,Enhanced-Ambition,South Korea,solar,PV,electricity,2035,0.528249
204,EJ,Enhanced-Ambition,South Korea,solar,PV_storage,electricity,2035,0.005032


In [93]:
282132 - 141804 + 282132

422460

In [94]:
df[(df['subsector'] == 'gas')]

,Units,scenario,region,subsector,technology,output,Year,value
26,EJ,Current-Policy,South Korea,gas,gas (CC CCS),electricity,2025,0.000638
27,EJ,Current-Policy,South Korea,gas,gas (CC CCS),electricity,2030,0.001748
28,EJ,Current-Policy,South Korea,gas,gas (CC CCS),electricity,2035,0.003124
29,EJ,Current-Policy,South Korea,gas,gas (CC H2 blend 50%),electricity,2025,0.014000
30,EJ,Current-Policy,South Korea,gas,gas (CC H2 blend 50%),electricity,2030,0.052024
31,EJ,Current-Policy,South Korea,gas,gas (CC H2 blend 50%),electricity,2035,0.111000
32,EJ,Current-Policy,South Korea,gas,gas (CC),electricity,1990,0.026982
33,EJ,Current-Policy,South Korea,gas,gas (CC),electricity,2005,0.202728
34,EJ,Current-Policy,South Korea,gas,gas (CC),electricity,2010,0.264734
35,EJ,Current-Policy,South Korea,gas,gas (CC),electricity,2015,0.429704


In [95]:
df[(df['subsector'] == 'hydro')]

,Units,scenario,region,subsector,technology,output,Year,value
48,EJ,Current-Policy,South Korea,hydro,hydro,electricity,1990,0.022900
49,EJ,Current-Policy,South Korea,hydro,hydro,electricity,2005,0.013223
50,EJ,Current-Policy,South Korea,hydro,hydro,electricity,2010,0.013255
51,EJ,Current-Policy,South Korea,hydro,hydro,electricity,2015,0.009511
52,EJ,Current-Policy,South Korea,hydro,hydro,electricity,2020,0.025700
53,EJ,Current-Policy,South Korea,hydro,hydro,electricity,2025,0.028800
54,EJ,Current-Policy,South Korea,hydro,hydro,electricity,2030,0.032800
55,EJ,Current-Policy,South Korea,hydro,hydro,electricity,2035,0.060100
157,EJ,Enhanced-Ambition,South Korea,hydro,hydro,electricity,1990,0.022900
158,EJ,Enhanced-Ambition,South Korea,hydro,hydro,electricity,2005,0.013223


In [96]:
df[(df['technology'].isin(['PV_storage', 'wind_storage']))].groupby(['scenario', 'Year'])['value'].sum()

scenario           Year
Current-Policy     2020    0.000245
                   2025    0.000498
                   2030    0.001071
                   2035    0.002761
Enhanced-Ambition  2020    0.000244
                   2025    0.000506
                   2030    0.001747
                   2035    0.008196
Name: value, dtype: float64

In [97]:
df[(df['technology'].isin(['refined liquids (CC CCS)', 'refined liquids (CC)', 'refined liquids (steam/CT)']))]#.groupby(['scenario', 'Year'])['value'].sum()

,Units,scenario,region,subsector,technology,output,Year,value
68,EJ,Current-Policy,South Korea,refined liquids,refined liquids (CC),electricity,2020,0.002703
69,EJ,Current-Policy,South Korea,refined liquids,refined liquids (CC),electricity,2025,0.005645
70,EJ,Current-Policy,South Korea,refined liquids,refined liquids (CC),electricity,2030,0.010380
71,EJ,Current-Policy,South Korea,refined liquids,refined liquids (CC),electricity,2035,0.021542
72,EJ,Current-Policy,South Korea,refined liquids,refined liquids (steam/CT),electricity,1990,0.067888
73,EJ,Current-Policy,South Korea,refined liquids,refined liquids (steam/CT),electricity,2005,0.075750
74,EJ,Current-Policy,South Korea,refined liquids,refined liquids (steam/CT),electricity,2010,0.053866
75,EJ,Current-Policy,South Korea,refined liquids,refined liquids (steam/CT),electricity,2015,0.038083
76,EJ,Current-Policy,South Korea,refined liquids,refined liquids (steam/CT),electricity,2020,0.022743
77,EJ,Current-Policy,South Korea,refined liquids,refined liquids (steam/CT),electricity,2025,0.027517


In [98]:
def catTech(x):
    if x in ['PV', 'PV_storage']:
        return 'Solar'
    elif x in ['wind', 'wind_offshore', 'wind_storage']:
        return 'Wind'
    elif x in ['refined liquids (CC)', 'refined liquids (steam/CT)']:
        return 'Oil'
    elif x in ['refined liquids (CC CCS)']:
        return 'Oil w/ CCS'
    elif x in ['hydro']:
        return 'Hydro'
    elif x in ['Gen_III', 'Gen_II_LWR']:
        return 'Nuclear'
    elif x in ['biomass (IGCC CCS)', 'biomass (conv CCS)']:
        return 'Biomass w/ CCS'
    elif x in ['biomass (IGCC)', 'biomass (conv)']:
        return 'Biomass'
    elif x in ['coal (conv pul ammonia blend 20%)']:
        return 'Ammonia'
    elif x in ['gas (CC H2 blend 50%)']:
        return 'Hydrogen'
    elif x in ['gas (CC CCS)']:
        return 'Gas w/ CCS'
    elif x in ['gas (CC)', 'gas (steam/CT)']:
        return 'Gas'
    elif x in ['coal (IGCC CCS)', 'coal (conv pul CCS)']:
        return 'Coal w/ CCS'
    elif x in ['coal (IGCC)', 'coal (conv pul)']:
        return 'Coal'

In [99]:
stack_order = [
    'Ammonia', 'Coal w/ CCS', 'Coal', 'Oil w/ CCS', 'Oil', 'Gas w/ CCS', 
    'Hydrogen', 'Gas',  'Nuclear', 'Biomass', 'Hydro', 'Wind', 'Solar'
]

In [100]:
df['genTech'] = df['technology'].apply(catTech)
df['genTech'] = pd.Categorical(df['genTech'], categories=stack_order, ordered=True)
df = df.sort_values(by=['Year', 'genTech'])
df['genTech'].unique()

['Coal', 'Oil', 'Gas', 'Nuclear', 'Hydro', ..., NaN, 'Coal w/ CCS', 'Gas w/ CCS', 'Hydrogen', 'Ammonia']
Length: 13
Categories (13, object): ['Ammonia' < 'Coal w/ CCS' < 'Coal' < 'Oil w/ CCS' ... 'Biomass' < 'Hydro' < 'Wind' < 'Solar']

In [101]:
df[(df['Year'] == 2035) & (df['genTech'] == 'Coal w/ CCS')]

,Units,scenario,region,subsector,technology,output,Year,value,genTech
12,EJ,Current-Policy,South Korea,coal,coal (IGCC CCS),electricity,2035,0.002798,Coal w/ CCS
15,EJ,Current-Policy,South Korea,coal,coal (conv pul CCS),electricity,2035,0.005984,Coal w/ CCS
123,EJ,Enhanced-Ambition,South Korea,coal,coal (IGCC CCS),electricity,2035,0.026832,Coal w/ CCS
126,EJ,Enhanced-Ambition,South Korea,coal,coal (conv pul CCS),electricity,2035,0.137920,Coal w/ CCS


In [102]:
df['value'] = df['value'].apply(ej_to_twh)
df['Units'] = 'TWh'

In [103]:
df[(df['technology'] == 'gas (CC H2 blend 50%)')]

,Units,scenario,region,subsector,technology,output,Year,value,genTech
29,TWh,Current-Policy,South Korea,gas,gas (CC H2 blend 50%),electricity,2025,3.888889,Hydrogen
138,TWh,Enhanced-Ambition,South Korea,gas,gas (CC H2 blend 50%),electricity,2025,3.889083,Hydrogen
30,TWh,Current-Policy,South Korea,gas,gas (CC H2 blend 50%),electricity,2030,14.451194,Hydrogen
139,TWh,Enhanced-Ambition,South Korea,gas,gas (CC H2 blend 50%),electricity,2030,23.271361,Hydrogen
31,TWh,Current-Policy,South Korea,gas,gas (CC H2 blend 50%),electricity,2035,30.833333,Hydrogen
140,TWh,Enhanced-Ambition,South Korea,gas,gas (CC H2 blend 50%),electricity,2035,52.361778,Hydrogen


In [104]:
df = df.sort_values(by=['Year', 'genTech'], ascending=True)

In [105]:
df[(df['genTech'] == 'Ammonia') & (df['Year'] == 2035)]

,Units,scenario,region,subsector,technology,output,Year,value,genTech
17,TWh,Current-Policy,South Korea,coal,coal (conv pul ammonia blend 20%),electricity,2035,86.944445,Ammonia


In [106]:
df[(df['genTech'] == 'Coal w/ CCS') & (df['Year'] == 2035)]

,Units,scenario,region,subsector,technology,output,Year,value,genTech
12,TWh,Current-Policy,South Korea,coal,coal (IGCC CCS),electricity,2035,0.777342,Coal w/ CCS
15,TWh,Current-Policy,South Korea,coal,coal (conv pul CCS),electricity,2035,1.662153,Coal w/ CCS
123,TWh,Enhanced-Ambition,South Korea,coal,coal (IGCC CCS),electricity,2035,7.453194,Coal w/ CCS
126,TWh,Enhanced-Ambition,South Korea,coal,coal (conv pul CCS),electricity,2035,38.311111,Coal w/ CCS


In [107]:
df[(df['genTech'] == 'Others') & (df['Year'] == 2035)]

,Units,scenario,region,subsector,technology,output,Year,value,genTech


In [108]:
dfFig = df[(df['Year'] >= 2015) & (df['Year'] <= 2035) &  (~df['genTech'].isna())].groupby(['scenario', 'Year', 'genTech'], observed=False)['value'].sum().reset_index()
dfFig

,scenario,Year,genTech,value
0,Current-Policy,2015,Ammonia,0.000000
1,Current-Policy,2015,Coal w/ CCS,0.000000
2,Current-Policy,2015,Coal,215.851667
3,Current-Policy,2015,Oil w/ CCS,0.000000
4,Current-Policy,2015,Oil,10.578611
...,...,...,...,...
125,Enhanced-Ambition,2035,Nuclear,235.833334
126,Enhanced-Ambition,2035,Biomass,10.631111
127,Enhanced-Ambition,2035,Hydro,16.694444
128,Enhanced-Ambition,2035,Wind,158.479932


In [109]:
# Pivot for easier calculation
pivot = dfFig.pivot_table(index=['scenario', 'Year'], columns='genTech', values='value', fill_value=0)

# Apply multipliers
pivot['Hydrogen'] = pivot['Hydrogen'] * 0.5
pivot['Ammonia'] = pivot['Ammonia'] * 0.2

# Add "remains" to Gas and Coal
pivot['Gas'] += pivot['Hydrogen']/0.5 * 0.5  # (original Hydrogen value * 0.5)
pivot['Coal'] += pivot['Ammonia']/0.2 * 0.8  # (original Ammonia value * 0.8)

# If needed, you can reset index
result_df = pivot.reset_index().melt(id_vars=['scenario', 'Year'], var_name='genTech', value_name='value')
result_df

/tmp/ipykernel_94225/1785598069.py:2: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior



,scenario,Year,genTech,value
0,Current-Policy,2015,Ammonia,0.000000
1,Current-Policy,2020,Ammonia,0.000000
2,Current-Policy,2025,Ammonia,0.000000
3,Current-Policy,2030,Ammonia,8.221667
4,Current-Policy,2035,Ammonia,17.388889
...,...,...,...,...
125,Enhanced-Ambition,2015,Solar,3.972611
126,Enhanced-Ambition,2020,Solar,22.028163
127,Enhanced-Ambition,2025,Solar,44.987989
128,Enhanced-Ambition,2030,Solar,92.965236


In [110]:
dfAgg1 = result_df[(result_df['scenario'] == scenarios[0])]
fig1 = px.bar(dfAgg1, x="Year", y="value", color="genTech", title="Current Policy", color_discrete_map=custom_colors)
dfAgg2 = result_df[(result_df['scenario'] == scenarios[1])]
fig2 = px.bar(dfAgg2, x="Year", y="value", color="genTech", title="Enhanced Ambition", color_discrete_map=custom_colors)

In [111]:
result_df['RE'] = result_df['genTech'].apply(lambda x: 1 if x in ['Solar', 'Wind', 'Hydro', 'Biomass'] else 0)
result_df['CF'] = result_df['genTech'].apply(lambda x: 1 if x in ['Solar', 'Wind', 'Hydro', 'Biomass', 'Nuclear'] else 0)
re_share_current = (result_df[(result_df['RE'] == 1) & (result_df['scenario'] == scenarios[0])].groupby(['Year'])['value'].sum() / result_df[(result_df['scenario'] == scenarios[0])].groupby(['Year'])['value'].sum()) * 100
cf_share_current = (result_df[(result_df['CF'] == 1) & (result_df['scenario'] == scenarios[0])].groupby(['Year'])['value'].sum() / result_df[(result_df['scenario'] == scenarios[0])].groupby(['Year'])['value'].sum()) * 100
re_share_enhanced = (result_df[(result_df['RE'] == 1) & (result_df['scenario'] == scenarios[1])].groupby(['Year'])['value'].sum() / result_df[(result_df['scenario'] == scenarios[1])].groupby(['Year'])['value'].sum()) * 100
cf_share_enhanced = (result_df[(result_df['CF'] == 1) & (result_df['scenario'] == scenarios[1])].groupby(['Year'])['value'].sum() / result_df[(result_df['scenario'] == scenarios[1])].groupby(['Year'])['value'].sum()) * 100

In [112]:
years = list(range(2015, 2036, 5))

# Create subplots with secondary y-axes
fig = make_subplots(
    rows=1, cols=2,
    shared_yaxes=True,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=("Current Policy", "Enhanced Ambition")
)
# === Legend Group: Gases ===

# # Add traces from fig1 to col 1
# for trace in fig1['data']:
#     fig.add_trace(trace, row=1, col=1, secondary_y=False)
for trace in fig1.data:
    trace.showlegend = False
    fig.add_trace(trace, row=1, col=1, secondary_y=False)
# Add RE/carbon-free share line to col 1
fig.add_trace(go.Scatter(
    x=years,
    y=re_share_current,  # % values
    name="RE Share",
    mode="markers",
    marker=dict(symbol='triangle-up', size=9, color="green"),
    showlegend=False
), row=1, col=1, secondary_y=True)

fig.add_trace(go.Scatter(
    x=years,
    y=cf_share_current,  # % values
    name="Carbon-Free Share",
    mode="markers",
    marker=dict(symbol='triangle-up', size=9, color="blue"), showlegend=False
), row=1, col=1, secondary_y=True)

# Add traces from fig5 to col 2
for trace in fig2['data']:
    fig.add_trace(trace, row=1, col=2, secondary_y=False)

# === Legend Group: Gases ===
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    name='<br><br><br><b>Technology</b>', showlegend=True, hoverinfo='skip'
))

# Add RE/carbon-free share to col 2
fig.add_trace(go.Scatter(
    x=years,
    y=re_share_enhanced,
    name="RE Share (%)",
    mode="markers",
    marker=dict(symbol='triangle-up', size=9, color="green"),
), row=1, col=2, secondary_y=True)

fig.add_trace(go.Scatter(
    x=years,
    y=cf_share_enhanced,
    name="Carbon-Free Share (%)",
    mode="markers",
    marker=dict(symbol='triangle-up', size=9, color="blue")
), row=1, col=2, secondary_y=True)

fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    line=dict(color='rgba(0,0,0,0)'),
    name='<b>Share</b>', showlegend=True, hoverinfo='skip'
))

fig.update_layout(
    yaxis=dict(title="Electricity Generation (TWh)", showgrid=True),
    yaxis1=dict(title="Electricity Generation (TWh)", showgrid=True, title_font_size=20),
    
    # Hide secondary y-axis for col 1 (still used internally)
    yaxis2=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    
    # Show secondary y-axis for col 2
    yaxis4=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        title='',
        overlaying='y',
        side='right',
        range=[0, 100]
    ),
    barmode='stack',
    plot_bgcolor='rgba(0,0,0,0)',
    width=800, height=700,
)

for i, year in enumerate(range(2015, 2036, 5)):
    
    fig.add_annotation(
        x=years[i],
        y=re_share_current[year] + 3,
        xref="x1", yref="y2",
        text=f"{re_share_current[year]:.0f}%",
        showarrow=False,
        arrowhead=2,
        ax=0, ay=-30,
        font=dict(color="green")
    )

    fig.add_annotation(
        x=years[i],
        y=cf_share_current[year] + 3,
        xref="x1", yref="y2",
        text=f"{cf_share_current[year]:.0f}%",
        showarrow=False,
        arrowhead=2,
        ax=0, ay=-30,
        font=dict(color="blue")
    )

    fig.add_annotation(
        x=years[i],
        y=re_share_enhanced[year] + 3,
        xref="x2", yref="y4",
        text=f"{re_share_enhanced[year]:.0f}%",
        showarrow=False,
        arrowhead=2,
        ax=0, ay=-30,
        font=dict(color="green")
    )

    fig.add_annotation(
        x=years[i],
        y=cf_share_enhanced[year] + 3,
        xref="x2", yref="y4",
        text=f"{cf_share_enhanced[year]:.0f}%",
        showarrow=False,
        arrowhead=2,
        ax=0, ay=-30,
        font=dict(color="blue")
    )

fig.update_xaxes(tickangle=45)

fig.update_layout(
    yaxis=dict(title="Electricity Generation (TWh)", showgrid=True, gridcolor='lightgray'),
    yaxis3=dict(showgrid=True, gridcolor='lightgray'),
    # title=dict(
        # text="<b>Electricity Generation and Clean Energy Share</b>",
        # font=dict(size=20),
        # x=0.5
    # ),
    legend=dict(
        traceorder="reversed",
        font=dict(size=15),
        x=1.02, y=1,
        borderwidth=0
    )
)

for i in range(1, 3):
    fig.update_xaxes(
        tickmode='array',
        tickangle=45,
        tickvals=list(range(2015, 2040, 5)),
        tickfont=dict(size=15),
        row=1, col=i
    )

fig.update_layout(
    yaxis=dict(
        title="Electricity Generation (TWh)",
        title_font=dict(size=18)  # 👈 controls y-axis label size
    )
)

# adjust axis labels and ticks
fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=18))

# bump the subplot titles
fig.update_annotations(font=dict(size=15))

# your subplot titles
subtitle_texts = ("Current Policy", "Enhanced Ambition")

fig.for_each_annotation(
    lambda ann: ann.update(
        # only bump font on those whose text matches one of your subtitles
        font=dict(size=21)
    ) if ann.text in subtitle_texts else None
)

fig.show()


In [56]:
448.7 / 691.5

0.648879248011569

In [43]:
(179.9+24.3) / 691.5

0.2953000723065799